In [1]:
# 1. Instalamos la librería para leer variables de entorno (.env)
!pip install python-dotenv requests pandas -q

import json
import os
from dotenv import load_dotenv
import pandas as pd
import requests

# --- CREACIÓN DE ARCHIVOS DE CONFIGURACIÓN (.env y .gitignore) ---
# Creamos el archivo .env con la configuración de la API (nunca se sube a GitHub)
with open(".env", "w") as f:
    f.write("API_URL=https://jsonplaceholder.typicode.com/users\n")
    f.write("API_TIMEOUT=10\n")

# Creamos el .gitignore para proteger el archivo .env
with open(".gitignore", "w") as f:
    f.write(".env\n__pycache__/\n.ipynb_checkpoints/\n")

# --- EXTRACCIÓN SEGURA DESDE LA API ---
# Cargamos las variables desde el archivo .env
load_dotenv()
url_api = os.getenv("API_URL")
tiempo_espera = int(os.getenv("API_TIMEOUT", 10))

print(f"Conectando de forma segura a: {url_api}...")

try:
    # Hacemos la petición GET con tiempo límite
    respuesta = requests.get(url_api, timeout=tiempo_espera)

    # Si la API devuelve un error (como 404 o 500), esto lanza una excepción
    respuesta.raise_for_status()

    # Convertimos la respuesta a formato JSON
    datos_crudos = respuesta.json()
    print(f"¡Éxito! Se descargaron {len(datos_crudos)} registros.")

    # --- TRANSFORMACIÓN BÁSICA ---
    # Limpiamos los datos dejando solo los campos que nos interesan
    datos_limpios = []
    for usuario in datos_crudos:
        datos_limpios.append({
            "id": usuario.get("id"),
            "nombre": usuario.get("name"),
            "usuario": usuario.get("username"),
            "email": usuario.get("email"),
            "ciudad": usuario.get("address", {}).get("city"),
            "empresa": usuario.get("company", {}).get("name"),
        })

    # --- PERSISTENCIA (GUARDADO) ---
    # Guardamos en formato CSV estructurado
    df = pd.DataFrame(datos_limpios)
    df.to_csv("data_extracted.csv", index=False, encoding="utf-8")

    print("Archivo 'data_extracted.csv' generado correctamente.\n")
    display(df.head())

except requests.exceptions.HTTPError as err_http:
    print(f"[Error HTTP] La API devolvió un código de fallo: {err_http}")
except requests.exceptions.ConnectionError:
    print(
        "[Error de Conexión] No se pudo conectar con el servidor de la API."
    )
except requests.exceptions.Timeout:
    print("[Error de Tiempo] La API tardó demasiado en responder.")
except Exception as err:
    print(f"[Error Inesperado] Ocurrió un problema: {err}")

Conectando de forma segura a: https://jsonplaceholder.typicode.com/users...
¡Éxito! Se descargaron 10 registros.
Archivo 'data_extracted.csv' generado correctamente.



,id,nombre,usuario,email,ciudad,empresa
0,1,Leanne Graham,Bret,Sincere@april.biz,Gwenborough,Romaguera-Crona
1,2,Ervin Howell,Antonette,Shanna@melissa.tv,Wisokyburgh,Deckow-Crist
2,3,Clementine Bauch,Samantha,Nathan@yesenia.net,McKenziehaven,Romaguera-Jacobson
3,4,Patricia Lebsack,Karianne,Julianne.OConner@kory.org,South Elvis,Robel-Corkery
4,5,Chelsey Dietrich,Kamren,Lucio_Hettinger@annie.ca,Roscoeview,Keebler LLC
